<a href="https://colab.research.google.com/github/navanathmp/DEEP-LEARNING/blob/main/Deep_Learning_Skill_Development_Tasks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning
### Skill Development Tasks (1 to 15) — Complete Practical Solutions

* **Student Name:** MP Navanath
* **Registration No.:** 2511022250012
* **Course:** Deep Learning


---
## Task 1: Vectorized Multi-Layer Perceptron (MLP) Forward & Backward Pass from Scratch
**Objective:** Implement a fully connected neural network with arbitrary hidden layer dimensions from scratch using pure NumPy matrix calculus.


In [ ]:
import numpy as np

class DenseLayer:
    def __init__(self, in_dim, out_dim):
        self.W = np.random.randn(in_dim, out_dim) * np.sqrt(2.0 / in_dim)
        self.b = np.zeros((1, out_dim))
        self.dW, self.db = None, None
        self.x = None

    def forward(self, x):
        self.x = x
        return np.dot(x, self.W) + self.b

    def backward(self, dout):
        N = self.x.shape[0]
        self.dW = np.dot(self.x.T, dout) / N
        self.db = np.sum(dout, axis=0, keepdims=True) / N
        return np.dot(dout, self.W.T)

class ScratchMLP:
    def __init__(self, layer_dims):
        self.layers = [DenseLayer(layer_dims[i], layer_dims[i+1]) for i in range(len(layer_dims) - 1)]

    def relu(self, z):
        return np.maximum(0, z)

    def softmax(self, z):
        exp_z = np.exp(z - np.max(z, axis=-1, keepdims=True))
        return exp_z / np.sum(exp_z, axis=-1, keepdims=True)

    def forward(self, x):
        self.activations = [x]
        out = x
        for layer in self.layers[:-1]:
            out = self.relu(layer.forward(out))
            self.activations.append(out)
        logits = self.layers[-1].forward(out)
        self.probs = self.softmax(logits)
        return self.probs

    def compute_loss(self, y_onehot):
        eps = 1e-12
        return -np.mean(np.sum(y_onehot * np.log(self.probs + eps), axis=1))

    def backward(self, y_onehot, lr=0.05):
        dout = self.probs - y_onehot
        dout = self.layers[-1].backward(dout)

        for i in reversed(range(len(self.layers) - 1)):
            drelu = dout * (self.activations[i+1] > 0)
            dout = self.layers[i].backward(drelu)

        for layer in self.layers:
            layer.W -= lr * layer.dW
            layer.b -= lr * layer.db

# Test Run
np.random.seed(42)
X = np.random.randn(64, 8)
y_idx = np.random.randint(0, 3, 64)
Y = np.eye(3)[y_idx]

model = ScratchMLP([8, 16, 16, 3])
for epoch in range(100):
    probs = model.forward(X)
    loss = model.compute_loss(Y)
    model.backward(Y, lr=0.1)
    if (epoch + 1) % 20 == 0:
        print(f"Task 1 | Epoch {epoch+1:03d} | Loss: {loss:.4f}")

---
## Task 2: Empirical Diagnostic of Custom Weight Initializations and Gradient Vanishing
**Objective:** Solve training instability by analyzing activation distributions and gradient magnitudes under varying initialization distributions (Xavier vs. Kaiming vs. Poor Small Init).


In [ ]:
import torch
import matplotlib.pyplot as plt

def xavier_init(in_dim, out_dim):
    return torch.randn(in_dim, out_dim) * (2.0 / (in_dim + out_dim)) ** 0.5

def kaiming_init(in_dim, out_dim):
    return torch.randn(in_dim, out_dim) * (2.0 / in_dim) ** 0.5

def bad_small_init(in_dim, out_dim):
    return torch.randn(in_dim, out_dim) * 0.01

def run_diagnostic(init_fn, num_layers=20, dim=128):
    x = torch.randn(500, dim)
    stds = []
    for _ in range(num_layers):
        W = init_fn(dim, dim)
        x = torch.relu(torch.matmul(x, W))
        stds.append(x.std().item())
    return stds

fig, ax = plt.subplots(figsize=(8, 4))
for name, fn in [("Bad Small Init (0.01)", bad_small_init),
                 ("Xavier (Glorot)", xavier_init),
                 ("Kaiming (He)", kaiming_init)]:
    stds = run_diagnostic(fn)
    print(f"[{name}] Layer 1 Std: {stds[0]:.4f} -> Layer 20 Std: {stds[-1]:.4e}")
    ax.plot(range(1, 21), stds, marker='o', label=name)

ax.set_title("Task 2: Activation Variance across 20 Layers")
ax.set_xlabel("Layer Depth")
ax.set_ylabel("Standard Deviation")
ax.set_yscale('log')
ax.grid(True)
ax.legend()
plt.show()

---
## Task 3: Vectorized Convolutional Forward Pass via im2col Transformation from Scratch
**Objective:** Implement a 2D convolutional layer supporting padding, strides, and input/output channels from scratch using `im2col`.


In [ ]:
import numpy as np

def im2col(x, kh, kw, padding=1, stride=1):
    N, C, H, W = x.shape
    out_h = (H + 2 * padding - kh) // stride + 1
    out_w = (W + 2 * padding - kw) // stride + 1
    x_padded = np.pad(x, ((0, 0), (0, 0), (padding, padding), (padding, padding)), mode='constant')

    cols = np.zeros((N, C, kh, kw, out_h, out_w))
    for i in range(kh):
        for j in range(kw):
            cols[:, :, i, j, :, :] = x_padded[:, :, i:i + out_h * stride:stride, j:j + out_w * stride:stride]
    cols = cols.transpose(1, 2, 3, 0, 4, 5).reshape(C * kh * kw, N * out_h * out_w)
    return cols, out_h, out_w

class VectorizedConv2D:
    def __init__(self, in_channels, out_channels, k_size=3, stride=1, padding=1):
        self.in_c, self.out_c = in_channels, out_channels
        self.k, self.stride, self.pad = k_size, stride, padding
        self.W = np.random.randn(out_channels, in_channels, k_size, k_size) * 0.05
        self.b = np.zeros((out_channels, 1))

    def forward(self, x):
        N, _, _, _ = x.shape
        x_col, out_h, out_w = im2col(x, self.k, self.k, self.pad, self.stride)
        w_row = self.W.reshape(self.out_c, -1)
        out = np.dot(w_row, x_col) + self.b
        return out.reshape(self.out_c, N, out_h, out_w).transpose(1, 0, 2, 3)

conv = VectorizedConv2D(in_channels=3, out_channels=16, k_size=3, stride=1, padding=1)
sample_in = np.random.randn(4, 3, 32, 32)
out = conv.forward(sample_in)
print(f"Task 3 | Input Shape: {sample_in.shape} -> Conv Output Shape: {out.shape}")

---
## Task 4: Custom Gradient Descent Optimizers with Momentum, RMSprop, and Adam
**Objective:** Implement optimizer update rules from scratch, calculating exponential moving averages of first and second moments with bias correction.


In [ ]:
import torch

class CustomAdam:
    def __init__(self, params, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params = list(params)
        self.lr, self.b1, self.b2, self.eps, self.t = lr, beta1, beta2, eps, 0
        self.m = [torch.zeros_like(p.data) for p in self.params]
        self.v = [torch.zeros_like(p.data) for p in self.params]

    def step(self):
        self.t += 1
        with torch.no_grad():
            for i, p in enumerate(self.params):
                if p.grad is None: continue
                g = p.grad
                self.m[i] = self.b1 * self.m[i] + (1 - self.b1) * g
                self.v[i] = self.b2 * self.v[i] + (1 - self.b2) * (g ** 2)
                m_hat = self.m[i] / (1.0 - self.b1 ** self.t)
                v_hat = self.v[i] / (1.0 - self.b2 ** self.t)
                p.data -= self.lr * m_hat / (torch.sqrt(v_hat) + self.eps)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

w = torch.tensor([10.0], requires_grad=True)
opt = CustomAdam([w], lr=0.1)
for epoch in range(60):
    opt.zero_grad()
    loss = (w - 3.5) ** 2
    loss.backward()
    opt.step()
print(f"Task 4 | Target: 3.5000 | Optimized Value: {w.item():.4f}")

---
## Task 5: Custom Batch Normalization and Layer Normalization Forward/Backward Propagation
**Objective:** Maximize training convergence speeds and derive manual gradient derivations through scale, shift, and normalize transformations.


In [ ]:
import numpy as np

class CustomBatchNorm1D:
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        self.eps, self.momentum = eps, momentum
        self.gamma = np.ones((1, num_features))
        self.beta = np.zeros((1, num_features))
        self.running_mean = np.zeros((1, num_features))
        self.running_var = np.ones((1, num_features))

    def forward(self, x, training=True):
        if training:
            N = x.shape[0]
            mu = np.mean(x, axis=0, keepdims=True)
            var = np.var(x, axis=0, keepdims=True)
            self.x_hat = (x - mu) / np.sqrt(var + self.eps)
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mu
            self.running_var = (1 - self.momentum) * self.running_var + self.momentum * var
            self.cache = (x, mu, var, self.x_hat)
            return self.gamma * self.x_hat + self.beta
        else:
            x_hat = (x - self.running_mean) / np.sqrt(self.running_var + self.eps)
            return self.gamma * x_hat + self.beta

    def backward(self, dout):
        x, mu, var, x_hat = self.cache
        N = x.shape[0]
        std_inv = 1.0 / np.sqrt(var + self.eps)
        self.dgamma = np.sum(dout * x_hat, axis=0, keepdims=True)
        self.dbeta = np.sum(dout, axis=0, keepdims=True)
        dx_hat = dout * self.gamma
        dvar = np.sum(dx_hat * (x - mu) * -0.5 * (std_inv ** 3), axis=0, keepdims=True)
        dmu = np.sum(dx_hat * -std_inv, axis=0, keepdims=True) + dvar * np.mean(-2.0 * (x - mu), axis=0, keepdims=True)
        return dx_hat * std_inv + dvar * 2.0 * (x - mu) / N + dmu / N

bn = CustomBatchNorm1D(num_features=4)
x_in = np.random.randn(8, 4) * 5 + 3
out = bn.forward(x_in)
dx = bn.backward(np.ones_like(out))
print(f"Task 5 | Output Mean: {out.mean():.4f}, Std: {out.std():.4f} | dx computed successfully!")

---
## Task 6: Bidirectional Long Short-Term Memory (BiLSTM) Cell Mechanics from Scratch
**Objective:** Implement gated recurrent equations and memory routing layouts from scratch without using `nn.LSTM`.


In [ ]:
import torch
import torch.nn as nn

class CustomLSTMCell(nn.Module):
    def __init__(self, in_dim, h_dim):
        super().__init__()
        self.h_dim = h_dim
        self.W_ih = nn.Parameter(torch.randn(4 * h_dim, in_dim) * 0.1)
        self.W_hh = nn.Parameter(torch.randn(4 * h_dim, h_dim) * 0.1)
        self.b = nn.Parameter(torch.zeros(4 * h_dim))

    def forward(self, x, state):
        h_prev, c_prev = state
        gates = torch.matmul(x, self.W_ih.T) + torch.matmul(h_prev, self.W_hh.T) + self.b
        i, f, c_cand, o = gates.chunk(4, dim=1)
        i, f, c_cand, o = torch.sigmoid(i), torch.sigmoid(f), torch.tanh(c_cand), torch.sigmoid(o)
        c_next = f * c_prev + i * c_cand
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

class CustomBiLSTM(nn.Module):
    def __init__(self, in_dim, h_dim):
        super().__init__()
        self.h_dim = h_dim
        self.fwd_cell = CustomLSTMCell(in_dim, h_dim)
        self.bwd_cell = CustomLSTMCell(in_dim, h_dim)

    def forward(self, seq_x):
        b, seq_len, _ = seq_x.shape
        h_f, c_f = torch.zeros(b, self.h_dim), torch.zeros(b, self.h_dim)
        fwd_outs = []
        for t in range(seq_len):
            h_f, c_f = self.fwd_cell(seq_x[:, t, :], (h_f, c_f))
            fwd_outs.append(h_f.unsqueeze(1))

        h_b, c_b = torch.zeros(b, self.h_dim), torch.zeros(b, self.h_dim)
        bwd_outs = []
        for t in reversed(range(seq_len)):
            h_b, c_b = self.bwd_cell(seq_x[:, t, :], (h_b, c_b))
            bwd_outs.insert(0, h_b.unsqueeze(1))

        return torch.cat([torch.cat(fwd_outs, dim=1), torch.cat(bwd_outs, dim=1)], dim=-1)

bilstm = CustomBiLSTM(in_dim=10, h_dim=32)
x = torch.randn(4, 12, 10)
print(f"Task 6 | Input: {x.shape} -> BiLSTM Output: {bilstm(x).shape}")

---
## Task 7: ResNet-50 Style Residual Bottleneck Block and Grouped Convolutions
**Objective:** Construct bottleneck projection networks and multi-channel grouped convolutions with identity shortcuts.


In [ ]:
import torch
import torch.nn as nn

class BottleneckBlock(nn.Module):
    expansion = 4
    def __init__(self, in_planes, planes, stride=1, groups=1):
        super().__init__()
        mid_planes = planes
        self.conv1 = nn.Conv2d(in_planes, mid_planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(mid_planes)
        self.conv2 = nn.Conv2d(mid_planes, mid_planes, kernel_size=3, stride=stride, padding=1, groups=groups, bias=False)
        self.bn2 = nn.BatchNorm2d(mid_planes)
        self.conv3 = nn.Conv2d(mid_planes, planes * self.expansion, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes * self.expansion)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes * self.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * self.expansion)
            )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.bn3(self.conv3(self.relu(self.bn2(self.conv2(self.relu(self.bn1(self.conv1(x)))))))) + self.shortcut(x))

block = BottleneckBlock(in_planes=64, planes=64, stride=1)
x = torch.randn(2, 64, 56, 56)
print(f"Task 7 | Input: {x.shape} -> ResNet Block Output: {block(x).shape}")

---
## Task 8: Semantic Segmentation Pipeline on Pixel-wise Grids using Custom Dice Loss
**Objective:** Build a U-Net architecture paired with a composite Soft Dice Loss and Binary Cross-Entropy.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        preds = torch.sigmoid(logits).view(-1)
        targets_f = targets.view(-1)
        intersection = (preds * targets_f).sum()
        dice = (2.0 * intersection + self.smooth) / (preds.sum() + targets_f.sum() + self.smooth)
        bce = F.binary_cross_entropy_with_logits(logits.view(-1), targets_f)
        return (1.0 - dice) + bce

class MiniUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.bottle = nn.Sequential(nn.Conv2d(16, 32, 3, padding=1), nn.ReLU())
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec = nn.Conv2d(32, 1, 3, padding=1)

    def forward(self, x):
        return self.dec(self.up(self.bottle(self.enc(x))))

model = MiniUNet()
criterion = DiceBCELoss()
img, mask = torch.randn(2, 3, 64, 64), (torch.rand(2, 1, 64, 64) > 0.7).float()
preds = model(img)
loss = criterion(preds, mask)
print(f"Task 8 | Prediction: {preds.shape} | Combined Dice-BCE Loss: {loss.item():.4f}")

---
## Task 9: Anchor-Free Object Detection (YOLOv8-style) Loss Backpropagation
**Objective:** Implement Complete Intersection over Union (CIoU) bounding box regression loss and multi-task loss tracking.


In [ ]:
import torch
import numpy as np

def bbox_ciou(box1, box2, eps=1e-7):
    b1_x1, b1_y1, b1_x2, b1_y2 = box1[..., 0], box1[..., 1], box1[..., 2], box1[..., 3]
    b2_x1, b2_y1, b2_x2, b2_y2 = box2[..., 0], box2[..., 1], box2[..., 2], box2[..., 3]

    inter = (torch.min(b1_x2, b2_x2) - torch.max(b1_x1, b2_x1)).clamp(0) *             (torch.min(b1_y2, b2_y2) - torch.max(b1_y1, b2_y1)).clamp(0)
    w1, h1 = (b1_x2 - b1_x1).clamp(0), (b1_y2 - b1_y1).clamp(0)
    w2, h2 = (b2_x2 - b2_x1).clamp(0), (b2_y2 - b2_y1).clamp(0)
    iou = inter / (w1 * h1 + w2 * h2 - inter + eps)

    cw = torch.max(b1_x2, b2_x2) - torch.min(b1_x1, b2_x1)
    ch = torch.max(b1_y2, b2_y2) - torch.min(b1_y1, b2_y1)
    c2 = cw ** 2 + ch ** 2 + eps

    b1_cx, b1_cy = (b1_x1 + b1_x2) / 2.0, (b1_y1 + b1_y2) / 2.0
    b2_cx, b2_cy = (b2_x1 + b2_x2) / 2.0, (b2_y1 + b2_y2) / 2.0
    rho2 = (b1_cx - b2_cx) ** 2 + (b1_cy - b2_cy) ** 2

    v = (4.0 / (np.pi ** 2)) * torch.pow(torch.atan(w2 / (h2 + eps)) - torch.atan(w1 / (h1 + eps)), 2)
    with torch.no_grad():
        alpha = v / (1.0 - iou + v + eps)

    return 1.0 - (iou - (rho2 / c2 + alpha * v))

pred_boxes = torch.tensor([[10.0, 10.0, 50.0, 50.0]], requires_grad=True)
true_boxes = torch.tensor([[12.0, 14.0, 52.0, 48.0]])
loss = bbox_ciou(pred_boxes, true_boxes).mean()
loss.backward()
print(f"Task 9 | CIoU Loss: {loss.item():.4f} | Box Gradients: {pred_boxes.grad[0].numpy()}")

---
## Task 10: Wasserstein GAN with Gradient Penalty (WGAN-GP) for Stable Image Synthesis
**Objective:** Implement WGAN-GP with 1-Lipschitz gradient penalty constraints.


In [ ]:
import torch
import torch.nn as nn
import torch.autograd as autograd

class Generator(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(latent_dim, 128), nn.LeakyReLU(0.2), nn.Linear(128, 28*28), nn.Tanh())
    def forward(self, z): return self.net(z).view(-1, 1, 28, 28)

class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(28*28, 128), nn.LeakyReLU(0.2), nn.Linear(128, 1))
    def forward(self, x): return self.net(x)

def compute_gp(critic, real, fake):
    alpha = torch.rand(real.size(0), 1, 1, 1)
    interpolates = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    d_inter = critic(interpolates)
    grad = autograd.grad(outputs=d_inter, inputs=interpolates,
                         grad_outputs=torch.ones_like(d_inter),
                         create_graph=True, retain_graph=True)[0].view(real.size(0), -1)
    return ((grad.norm(2, dim=1) - 1) ** 2).mean()

gen, critic = Generator(), Critic()
real, fake = torch.randn(8, 1, 28, 28), gen(torch.randn(8, 64))
gp = compute_gp(critic, real, fake)
print(f"Task 10 | Critic Wasserstein Loss: {(-critic(real).mean() + critic(fake).mean() + 10.0 * gp).item():.4f}")

---
## Task 11: Vision Transformer (ViT) Patch Projection and Multi-Head Self-Attention from Scratch
**Objective:** Construct a Vision Transformer block from first principles using `einops`.


In [ ]:
!pip install -q einops

import torch
import torch.nn as nn
from einops import rearrange

class ViTPatchEmbedding(nn.Module):
    def __init__(self, in_c=3, patch_size=16, emb_dim=128, img_size=64):
        super().__init__()
        self.proj = nn.Conv2d(in_c, emb_dim, kernel_size=patch_size, stride=patch_size)
        num_patches = (img_size // patch_size) ** 2
        self.cls = nn.Parameter(torch.randn(1, 1, emb_dim))
        self.pos = nn.Parameter(torch.randn(1, num_patches + 1, emb_dim))

    def forward(self, x):
        b = x.shape[0]
        x = self.proj(x).flatten(2).transpose(1, 2)
        return torch.cat([self.cls.expand(b, -1, -1), x], dim=1) + self.pos

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, emb_dim=128, heads=4):
        super().__init__()
        self.heads, self.scale = heads, (emb_dim // heads) ** -0.5
        self.qkv = nn.Linear(emb_dim, emb_dim * 3)
        self.proj = nn.Linear(emb_dim, emb_dim)

    def forward(self, x):
        q, k, v = [rearrange(t, 'b n (h d) -> b h n d', h=self.heads) for t in self.qkv(x).chunk(3, dim=-1)]
        attn = torch.softmax(torch.matmul(q, k.transpose(-2, -1)) * self.scale, dim=-1)
        return self.proj(rearrange(torch.matmul(attn, v), 'b h n d -> b n (h d)'))

patch_emb = ViTPatchEmbedding()
mha = MultiHeadSelfAttention()
tokens = patch_emb(torch.randn(2, 3, 64, 64))
out = mha(tokens)
print(f"Task 11 | ViT Tokens: {tokens.shape} -> Attention Output: {out.shape}")

---
## Task 12: High-Dimensional Latent Space Anomaly Detection via Convolutional Autoencoders
**Objective:** Identify micro-structural patterns and outliers by mapping reconstruction error fields within bottlenecks.


In [ ]:
import torch
import torch.nn as nn

class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.ReLU(),
                                 nn.Conv2d(16, 8, 3, stride=2, padding=1), nn.ReLU())
        self.dec = nn.Sequential(nn.ConvTranspose2d(8, 16, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),
                                 nn.ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1), nn.Sigmoid())
    def forward(self, x): return self.dec(self.enc(x))

def detect_anomalies(model, test_img, threshold=0.08):
    model.eval()
    with torch.no_grad():
        recon = model(test_img)
        err_map = (test_img - recon) ** 2
        anomalies = err_map > threshold
    return recon, anomalies, err_map.mean(dim=[1,2,3])

ae = ConvAutoencoder()
normal_sample = torch.rand(4, 1, 28, 28)
recon, mask, scores = detect_anomalies(ae, normal_sample)
print(f"Task 12 | Input: {normal_sample.shape} -> Recon: {recon.shape} | Anomaly Score: {scores[0].item():.4f}")

---
## Task 13: Temporal Convolutional Networks (TCN) with Dilated Causal Convolutions
**Objective:** Model time-series sequences with dilated causal convolutions and zero lookahead leak.


In [ ]:
import torch
import torch.nn as nn

class CausalConv1d(nn.Module):
    def __init__(self, in_c, out_c, k=3, dilation=1):
        super().__init__()
        self.pad = (k - 1) * dilation
        self.conv = nn.Conv1d(in_c, out_c, k, padding=self.pad, dilation=dilation)

    def forward(self, x):
        out = self.conv(x)
        return out[:, :, :-self.pad] if self.pad > 0 else out

class TCN(nn.Module):
    def __init__(self, in_c=1, num_channels=[16, 16, 16]):
        super().__init__()
        layers = []
        for i, ch in enumerate(num_channels):
            in_ch = in_c if i == 0 else num_channels[i-1]
            layers.extend([CausalConv1d(in_ch, ch, k=3, dilation=2**i), nn.ReLU()])
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

tcn = TCN(in_c=1, num_channels=[16, 16, 16])
x_ts = torch.randn(4, 1, 100)
print(f"Task 13 | Input TimeSeries: {x_ts.shape} -> TCN Output: {tcn(x_ts).shape}")

---
## Task 14: Custom CUDA-Accelerated Swish Activation Layer Integration
**Objective:** Build custom CUDA kernels to accelerate non-linear operations on GPU.


In [ ]:
import torch
from torch.utils.cpp_extension import load_inline

cuda_source = '''
__global__ void swish_forward_kernel(const float* __restrict__ x, float* __restrict__ out, float beta, int size) {
    int idx = blockDim.x * blockIdx.x + threadIdx.x;
    if (idx < size) {
        float sig = 1.0f / (1.0f + expf(-beta * x[idx]));
        out[idx] = x[idx] * sig;
    }
}

torch::Tensor swish_forward_cuda(torch::Tensor x, float beta) {
    auto out = torch::empty_like(x);
    int size = x.numel();
    const int threads = 256;
    const int blocks = (size + threads - 1) / threads;
    swish_forward_kernel<<<blocks, threads>>>(x.data_ptr<float>(), out.data_ptr<float>(), beta, size);
    return out;
}
'''

cpp_source = "torch::Tensor swish_forward_cuda(torch::Tensor x, float beta);"

if torch.cuda.is_available():
    custom_swish = load_inline(
        name="custom_swish",
        cpp_sources=cpp_source,
        cuda_sources=cuda_source,
        functions=["swish_forward_cuda"],
        verbose=False
    )
    x = torch.randn(1000, device='cuda')
    out = custom_swish.swish_forward_cuda(x, 1.0)
    print(f"Task 14 | CUDA Swish executed on GPU! Sample: {out[:3].cpu().numpy()}")
else:
    x = torch.randn(1000)
    out = x * torch.sigmoid(x)
    print(f"Task 14 (CPU Fallback) | Swish Output: {out[:3].numpy()}")

---
## Task 15: Distributed Data Parallel (DDP) Multi-GPU Node Sync Gateway with GLOO/NCCL
**Objective:** Master distributed deep learning training loops and state synchronization across nodes.


In [ ]:
import os
import torch
import torch.nn as nn
import torch.distributed as dist
import torch.multiprocessing as mp

def run_worker(rank, world_size):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    dist.init_process_group("gloo", rank=rank, world_size=world_size)

    model = nn.Linear(8, 2)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

    x = torch.randn(16, 8)
    y = torch.randint(0, 2, (16,))

    optimizer.zero_grad()
    loss = nn.CrossEntropyLoss()(model(x), y)
    loss.backward()

    for param in model.parameters():
        dist.all_reduce(param.grad.data, op=dist.ReduceOp.SUM)
        param.grad.data /= world_size

    optimizer.step()
    if rank == 0:
        print(f"Task 15 | DDP Sync Successful across {world_size} worker nodes! Loss: {loss.item():.4f}")
    dist.destroy_process_group()

if __name__ == '__main__':
    world_size = 2
    mp.spawn(run_worker, args=(world_size,), nprocs=world_size, join=True)